# LP–GP Matching System — Vineyard Ventures

**Objective:** Automatically identify and rank the top LPs from Vineyard's CRM for a specific GP fund opportunity.

## GP Opportunity Context
| Field | Value |
|-------|-------|
| Fund | Vineyard Ventures |
| Fund size | ~$20M (very small / emerging fund) |
| Strategy | Fund-of-Funds (FoF) — invests in GPs, not companies directly |
| Focus | Pre-seed and seed stage |
| Geography | Indian deeptech — AI, hardware, defence, biotech |
| Team | Young, London-based, first-time GP (Fund 1) |

## Pipeline
```
Notion CRM  →  fetch_notion_data.py  →  data/raw_lp_data.json
                                    ↓
                       fetch_linkedin.py  →  data/linkedin_enrichment.json
                                    ↓
                      extract_signals.py  →  data/lp_signals.json
                                    ↓
                  lp_gp_matching.ipynb  →  ranked shortlist + visualisations
```

## Notebook Sections
1. **Ranked Shortlist** — full LP table sorted by composite fit score
2. **Signal Visualisations** — bar charts and lens weight comparison
3. **Dimension Breakdown** — radar charts (top 6) and cross-lens sensitivity analysis
4. **Tier 1 & 2 LP Profiles** — detailed evidence for each priority LP
5. **Outreach Recommendations** — concrete next actions and key messaging per LP
6. **LPs to Deprioritise** — rationale for Tier 3 and Tier 4 LPs
7. **Summary Statistics** — portfolio-level numbers


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 50)

# ── Load signals ───────────────────────────────────────────────────────────
signals_path = Path('data/lp_signals.json')
raw_path     = Path('data/raw_lp_data.json')

with open(signals_path) as f:
    records = json.load(f)

with open(raw_path) as f:
    raw_lps = json.load(f)

# Build word-count lookup for data quality (keyed by LP name)
note_wordcount = {r['name']: len(r.get('call_notes', '').split()) for r in raw_lps}

print(f'Loaded {len(records)} LP signal records')
print(f'Note coverage: {sum(1 for w in note_wordcount.values() if w > 50)}/{len(note_wordcount)} LPs have >50-word notes')

In [ ]:
# ── Configurable Weight Lenses ─────────────────────────────────────────────
# Each lens is a named weighting strategy. Weights must sum to 1.0.
# Change DEFAULT_LENS to score LPs differently for a different GP type.

LENSES = {
    "deeptech_india": {
        "label":       "Deeptech India (default)",
        "description": "Tuned for this GP — first-time Indian deeptech fund",
        "rationale": {
            "fit_score":      "AI's holistic fit assessment anchors the score",
            "india_interest": "India/EM exposure is a hard filter for this fund",
            "fof_openness":   "Must be willing to invest via a fund vehicle",
            "engagement":     "Warm LPs close faster",
            "emerging_mgr":   "First-time fund is hardest sell — bonus for LPs open to it",
            "deeptech":       "Nice-to-have: deeptech experience reduces education burden",
        },
        "weights": dict(fit_score=0.25, india_interest=0.25, fof_openness=0.20,
                        engagement=0.15, emerging_mgr=0.10, deeptech=0.05),
    },
    "sector_first": {
        "label":       "Sector-First",
        "description": "For niche sector funds where thematic alignment is paramount",
        "rationale":   {},
        "weights": dict(fit_score=0.25, india_interest=0.10, fof_openness=0.20,
                        engagement=0.15, emerging_mgr=0.10, deeptech=0.20),
    },
    "geography_first": {
        "label":       "Geography-First",
        "description": "For EM funds where geographic fit is the primary filter",
        "rationale":   {},
        "weights": dict(fit_score=0.15, india_interest=0.35, fof_openness=0.20,
                        engagement=0.15, emerging_mgr=0.10, deeptech=0.05),
    },
    "engagement_first": {
        "label":       "Engagement-First",
        "description": "Pipeline velocity — prioritise warm LPs most likely to close soon",
        "rationale":   {},
        "weights": dict(fit_score=0.20, india_interest=0.15, fof_openness=0.15,
                        engagement=0.35, emerging_mgr=0.10, deeptech=0.05),
    },
    "balanced": {
        "label":       "Balanced",
        "description": "Equal weighting — for generalist funds with no strong bias",
        "rationale":   {},
        "weights": dict(fit_score=0.20, india_interest=0.20, fof_openness=0.20,
                        engagement=0.20, emerging_mgr=0.10, deeptech=0.10),
    },
}

DEFAULT_LENS = "deeptech_india"   # ← change this to switch the primary ranking

# ── Scoring function ───────────────────────────────────────────────────────
def compute_composite(row: pd.Series, lens_name: str) -> float:
    w = LENSES[lens_name]["weights"]
    return round(
        w["fit_score"]      * row["Fit Score"] +
        w["india_interest"] * row["India Interest"] +
        w["fof_openness"]   * row["FoF Openness"] +
        w["engagement"]     * row["Engagement"] +
        w["emerging_mgr"]   * (10 if row["Emerging Mgr"] else 0) +
        w["deeptech"]       * (10 if row["Deeptech"] else 0),
        1
    )

# ── Build flat DataFrame ───────────────────────────────────────────────────
rows = []
for r in records:
    s  = r['signals']
    sf = r.get('structured_fields', {})
    rows.append({
        'Name':               r['name'],
        'Status':             sf.get('Status', '—'),
        'Location':           sf.get('Location', '—'),
        'LP Type':            s.get('lp_type', '—'),
        'AUM':                s.get('aum_estimate') or '—',
        'Check Size':         s.get('typical_check_size') or sf.get('Check Size', '—'),
        'Fit Score':          s.get('fit_score', 0),
        'India Interest':     s.get('india_interest', 0),
        'FoF Openness':       s.get('fof_openness', 0),
        'Engagement':         s.get('engagement_level', 0),
        'FoF Experience':     s.get('fof_experience', False),
        'Emerging Mgr':       s.get('emerging_manager_preference', False),
        'Deeptech':           s.get('deeptech_interest', False),
        'Sector Agnostic':    s.get('sector_agnostic', False),
        'Preferred Fund Size':s.get('preferred_fund_size') or '—',
        'Fit Rationale':      s.get('fit_rationale', ''),
        'Blockers':           '; '.join(s.get('blockers', [])),
        'Positives':          '; '.join(s.get('positives', [])),
        'Notion URL':         r.get('notion_url', ''),
        'Note Words':         note_wordcount.get(r['name'], 0),
    })

df = pd.DataFrame(rows)

# Compute composite under default lens, then sort by it
df['Composite'] = df.apply(lambda row: compute_composite(row, DEFAULT_LENS), axis=1)
df = df.sort_values('Composite', ascending=False).reset_index(drop=True)
df.index += 1   # 1-based rank (rank = position under the selected lens)

# Tier labels
def tier(score):
    if score >= 7.5: return 'Tier 1 — Priority'
    if score >= 6.0: return 'Tier 2 — Strong'
    if score >= 4.5: return 'Tier 3 — Moderate'
    return 'Tier 4 — Low'

df['Tier'] = df['Composite'].apply(tier)

print(f'Rankings computed under lens: "{LENSES[DEFAULT_LENS]["label"]}"')
print()
df[['Name', 'LP Type', 'Fit Score', 'India Interest', 'FoF Openness', 'Engagement', 'Composite', 'Tier']]

## 1. Ranked Shortlist

All 13 LPs ranked by their **composite fit score** under the active lens (`DEFAULT_LENS`).  
Change `DEFAULT_LENS` at the top of the scoring cell and re-run to instantly re-rank.  

| Tier | Score range | Meaning |
|------|-------------|------------------------------------------|
| Tier 1 — Priority | ≥ 7.5 | Strong across all dimensions; pursue now |
| Tier 2 — Strong   | ≥ 6.0 | Good fit; worth a follow-up meeting |
| Tier 3 — Moderate | ≥ 4.5 | Potential with caveats; nurture pipeline |
| Tier 4 — Low      | < 4.5 | Strategic misalignment; deprioritise |


In [ ]:
display_cols = ['Name', 'Tier', 'Status', 'LP Type', 'AUM', 'Check Size',
                'Fit Score', 'India Interest', 'FoF Openness', 'Engagement', 'Composite']

tier_colors = {
    'Tier 1 — Priority': 'background-color: #d4edda',
    'Tier 2 — Strong':   'background-color: #d1ecf1',
    'Tier 3 — Moderate': 'background-color: #fff3cd',
    'Tier 4 — Low':      'background-color: #f8d7da',
}

def color_tier(row):
    color = tier_colors.get(row['Tier'], '')
    return [color] * len(row)

print(f'Full LP ranking — lens: "{LENSES[DEFAULT_LENS]["label"]}"')
print(f'Sorted by Composite score (highest = best fit for this GP)\n')
df[display_cols].style.apply(color_tier, axis=1)

## 2. Signal Visualisations

**Left chart:** Composite score per LP, coloured by tier.  
**Right chart:** How each LP scores across the four key dimensions (Fit Score, India Interest, FoF Openness, Engagement).  

Below the charts: a **lens weight comparison table** showing how the five scoring presets differ.
The active lens is highlighted in green.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Composite score bar chart ───────────────────────────────────────
ax = axes[0]
names = df['Name'].tolist()
composites = df['Composite'].tolist()
tier_list = df['Tier'].tolist()

color_map = {
    'Tier 1 — Priority': '#2ecc71',
    'Tier 2 — Strong':   '#3498db',
    'Tier 3 — Moderate': '#f39c12',
    'Tier 4 — Low':      '#e74c3c',
}
bar_colors = [color_map[t] for t in tier_list]

bars = ax.barh(names[::-1], composites[::-1], color=bar_colors[::-1], edgecolor='white', height=0.7)
ax.set_xlabel('Composite Score (0–10)', fontsize=11)
ax.set_title('LP Composite Fit Score\n(40% fit · 20% India · 20% FoF · 20% engagement)', fontsize=11)
ax.set_xlim(0, 10)
ax.axvline(7.5, color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1.2, label='Tier 1 threshold')
ax.axvline(6.0, color='#3498db', linestyle='--', alpha=0.6, linewidth=1.2, label='Tier 2 threshold')
ax.axvline(4.5, color='#f39c12', linestyle='--', alpha=0.6, linewidth=1.2, label='Tier 3 threshold')
for bar, score in zip(bars[::-1], composites[::-1]):
    ax.text(score + 0.1, bar.get_y() + bar.get_height() / 2, f'{score}',
            va='center', fontsize=9)
patches = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
ax.legend(handles=patches, loc='lower right', fontsize=8)

# ── Right: Heatmap of key signal dimensions ────────────────────────────────
ax2 = axes[1]
dims = ['India\nInterest', 'FoF\nOpenness', 'Engagement', 'Fit\nScore']
data_cols = ['India Interest', 'FoF Openness', 'Engagement', 'Fit Score']
heat_data = df[data_cols].values.astype(float)

im = ax2.imshow(heat_data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=10)
ax2.set_xticks(range(len(dims)))
ax2.set_xticklabels(dims, fontsize=10)
ax2.set_yticks(range(len(names)))
ax2.set_yticklabels(names, fontsize=9)
ax2.set_title('Signal Heatmap by Dimension', fontsize=11)
plt.colorbar(im, ax=ax2, label='Score (0–10)')

for i in range(len(names)):
    for j in range(len(dims)):
        val = heat_data[i, j]
        ax2.text(j, i, f'{int(val)}', ha='center', va='center',
                 fontsize=9, color='black' if 3 < val < 8 else 'white')

plt.tight_layout()
plt.savefig('data/lp_signal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/lp_signal_analysis.png')

In [ ]:
# ── Display all lenses as a comparison table ───────────────────────────────
lens_rows = []
for key, lens in LENSES.items():
    w = lens["weights"]
    row = {"Lens": lens["label"], "Description": lens["description"]}
    row.update({k.replace("_", " ").title(): f"{v*100:.0f}%" for k, v in w.items()})
    row["Active"] = "✓" if key == DEFAULT_LENS else ""
    lens_rows.append(row)

lens_df = pd.DataFrame(lens_rows).set_index("Lens")

def highlight_active(row):
    return ['font-weight: bold; background-color: #e8f4e8' if row.name == LENSES[DEFAULT_LENS]["label"]
            else '' for _ in row]

print("All scoring lenses (active lens highlighted):")
lens_df.style.apply(highlight_active, axis=1)

## 3. Dimension Breakdown per LP

**Radar charts** for the top-6 LPs showing relative strength across four scoring dimensions.  
Each axis is 0–10; a larger area means stronger alignment with this GP opportunity.

**Sensitivity analysis** below shows each LP's rank under *every* lens — LPs that appear in the top 5
across multiple lenses are robust picks regardless of scoring assumptions.


In [ ]:
# Spider/radar chart for top-6 LPs
top6 = df.head(6)
categories = ['Fit Score', 'India Interest', 'FoF Openness', 'Engagement']
N = len(categories)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the loop

fig, axes = plt.subplots(2, 3, figsize=(14, 9), subplot_kw=dict(polar=True))
axes = axes.flatten()

colors = ['#2ecc71', '#2ecc71', '#3498db', '#3498db', '#f39c12', '#f39c12']

for idx, (_, row) in enumerate(top6.iterrows()):
    values = [row[c] for c in categories]
    values += values[:1]
    ax = axes[idx]
    ax.plot(angles, values, color=colors[idx], linewidth=2)
    ax.fill(angles, values, color=colors[idx], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 10)
    ax.set_yticks([2, 4, 6, 8, 10])
    ax.set_yticklabels(['2','4','6','8','10'], fontsize=6, color='grey')
    ax.set_title(f"{row['Name']}\n(composite: {row['Composite']})",
                 fontsize=10, pad=12)

plt.suptitle('Top-6 LP Profiles — Signal Radar', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('data/lp_radar_top6.png', dpi=150, bbox_inches='tight')
plt.show()
print('Radar chart saved to data/lp_radar_top6.png')

In [ ]:
# ── Compute rankings under every lens ─────────────────────────────────────
sensitivity = pd.DataFrame({"Name": df["Name"].values})

for lens_key, lens_meta in LENSES.items():
    scores = df.apply(lambda row: compute_composite(row, lens_key), axis=1)
    ranked = scores.rank(ascending=False, method='min').astype(int)
    col_label = lens_meta["label"].split(" (")[0]  # short name
    sensitivity[col_label] = ranked.values

sensitivity = sensitivity.set_index("Name")

# Highlight the default-lens column
default_col = LENSES[DEFAULT_LENS]["label"].split(" (")[0]

def highlight_default(col):
    return ['font-weight: bold; background-color: #e8f4e8'
            if col.name == default_col else '' for _ in col]

# Colour cells: green for rank 1-5, yellow 6-8, red 9+
def color_rank(val):
    if val <= 5:  return 'background-color: #c6efce; color: #276221'
    if val <= 8:  return 'background-color: #ffeb9c; color: #9c6500'
    return 'background-color: #ffc7ce; color: #9c0006'

# Sort rows by default lens ranking
sensitivity = sensitivity.sort_values(default_col)

print("LP rank under each scoring lens  (1 = best fit, green = top 5, red = bottom tier)\n")
sensitivity.style.apply(highlight_default).applymap(color_rank)


# ── Text summary of biggest movers ────────────────────────────────────────
print("\nBiggest rank changes across lenses:")
sensitivity["Range"] = sensitivity.max(axis=1) - sensitivity.min(axis=1)
for lp, row in sensitivity.sort_values("Range", ascending=False).head(4).iterrows():
    best_lens  = row.drop("Range").idxmin()
    worst_lens = row.drop("Range").idxmax()
    print(f"  {lp}: ranks #{int(row[best_lens])} under '{best_lens}' → #{int(row[worst_lens])} under '{worst_lens}'")

## 4. Tier 1 & 2 LP Profiles — Deep Dive

Detailed profile for every LP in Tier 1 (Priority) and Tier 2 (Strong).  
Each entry surfaces the specific evidence — from structured CRM fields or verbatim call notes —
that drove the ranking, along with identified blockers and positives.


In [ ]:
priority_lps = df[df['Tier'].isin(['Tier 1 — Priority', 'Tier 2 — Strong'])]

for rank, row in priority_lps.iterrows():
    tier_badge = {'Tier 1 — Priority': '🟢', 'Tier 2 — Strong': '🔵'}.get(row['Tier'], '')
    print(f"{'='*72}")
    print(f"#{rank}  {tier_badge}  {row['Name']}  ({row['LP Type']})  |  Composite: {row['Composite']}/10")
    print(f"   Status: {row['Status']}  |  Location: {row['Location']}  |  AUM: {row['AUM']}  |  Check: {row['Check Size']}")
    print(f"   Fit: {row['Fit Score']}/10  |  India: {row['India Interest']}/10  |  FoF: {row['FoF Openness']}/10  |  Engagement: {row['Engagement']}/10")
    print(f"   FoF Exp: {row['FoF Experience']}  |  Emerging-mgr preference: {row['Emerging Mgr']}  |  Deeptech: {row['Deeptech']}  |  Sector-agnostic: {row['Sector Agnostic']}")
    print(f"   Pref fund size: {row['Preferred Fund Size']}")
    print(f"\n   RATIONALE: {row['Fit Rationale']}")
    print(f"\n   POSITIVES: {row['Positives']}")
    if row['Blockers']:
        print(f"   BLOCKERS:  {row['Blockers']}")
    print(f"\n   Notion: {row['Notion URL']}")
    print()

print('='*72)

## 5. Outreach Recommendations

Concrete next actions and tailored key messages for each priority LP.  
Urgency reflects both tier and current engagement warmth from the CRM.


In [ ]:
recommendations = [
    {
        'Rank': 1,
        'LP': 'Dziugas',
        'Tier': 'Tier 1',
        'Urgency': 'High — anchor candidate',
        'Next Action': 'Send working doc outlining LP involvement model + GP co-access rights. Follow up end of Jan when exit process closes.',
        'Key Message': 'First-believer framing — small intimate fund where he calls me, not a faceless institution. Anchor $5–10M post-exit.',
    },
    {
        'Rank': 2,
        'LP': 'GEM',
        'Tier': 'Tier 1',
        'Urgency': 'High — in diligence, warm to India',
        'Next Action': 'Provide India benchmarking: intro to Prime, Sauce, Northpoint. Follow up with differentiated emerging GP case study.',
        'Key Message': 'Position Vineyard as the missing India mid-tier access layer. Emphasise consistency, brand-building, founder references.',
    },
    {
        'Rank': 3,
        'LP': 'Alber blanc',
        'Tier': 'Tier 1',
        'Urgency': 'Medium — Qualified, responsive',
        'Next Action': 'Arrange intro call / meeting in NYC. Share deck and emerging manager track record comp.',
        'Key Message': 'Validate India thesis with simple narrative. Lean on his openness to first-time GPs and FoF experience.',
    },
    {
        'Rank': 4,
        'LP': 'Weizmann Institute endowment',
        'Tier': 'Tier 2',
        'Urgency': 'Medium — In Nurture, deeptech + India exposure',
        'Next Action': 'Arrange LP meeting with detailed India deeptech landscape. Address valuation and DPI concerns with exit data from portfolio GPs.',
        'Key Message': 'Fund 1 specialist — ideal entry point. Differentiated mandate vs Z47/Accel. Concrete DPI narrative from GP portfolio.',
    },
    {
        'Rank': 5,
        'LP': 'Rezayat',
        'Tier': 'Tier 2',
        'Urgency': 'Low-Medium — Nurture, FoF + EM experience',
        'Next Action': 'Leverage James Eggington relationship. Build India interest with deeptech angle given industrial background.',
        'Key Message': 'Industrial / defence deeptech alignment. Relationship-driven, small team — play up intimacy of fund model.',
    },
]

rec_df = pd.DataFrame(recommendations).set_index('Rank')
rec_df.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'})

## 6. LPs to Deprioritise

Tier 3 LPs warrant a nurture-only approach — no near-term ask.  
Tier 4 LPs have clear structural misalignment with this GP; time is better spent elsewhere.


In [ ]:
low_tier = df[df['Tier'] == 'Tier 4 — Low'][['Name', 'LP Type', 'Composite', 'Fit Rationale']]
print('Tier 4 LPs — Low priority / strategic misalignment:')
display(low_tier)

print('\nTier 3 LPs — Moderate fit, nurture only:')
moderate = df[df['Tier'] == 'Tier 3 — Moderate'][['Name', 'LP Type', 'Composite', 'Blockers']]
display(moderate)

## 7. Summary Statistics

Portfolio-level summary: tier distribution, score averages, and CRM status breakdown.


In [ ]:
print('=== VINEYARD VENTURES — LP SHORTLIST SUMMARY ===')
print(f'Total LPs analysed: {len(df)}')
print()
tier_counts = df['Tier'].value_counts().sort_index()
for tier, count in tier_counts.items():
    print(f'  {tier}: {count} LP(s)')
print()
print(f'Average composite score: {df["Composite"].mean():.1f}/10')
print(f'Average India interest:  {df["India Interest"].mean():.1f}/10')
print(f'Average FoF openness:    {df["FoF Openness"].mean():.1f}/10')
print(f'Average engagement:      {df["Engagement"].mean():.1f}/10')
print()
print(f'LPs with FoF experience:          {df["FoF Experience"].sum()}/{len(df)}')
print(f'LPs preferring emerging managers: {df["Emerging Mgr"].sum()}/{len(df)}')
print(f'LPs with deeptech interest:       {df["Deeptech"].sum()}/{len(df)}')
print()
print('Status breakdown:')
status_breakdown = df.groupby('Status')['Name'].apply(list)
for status, names in status_breakdown.items():
    print(f'  {status}: {", ".join(names)}')